# 03 — Comparing Activation Functions: ReLU vs Sigmoid vs Tanh

**Objective:** Experiment with ReLU, Sigmoid, and Tanh activation functions and compare their effects on neural network training on MNIST.
All other hyperparameters (architecture, optimizer, batch size, epochs, data split) are kept identical.

---
## 1. Concepts: Activation Functions

### ReLU (Rectified Linear Unit)
```
f(x) = max(0, x)
Range: [0, +inf)
```
- Fast to compute; does not saturate for positive values.
- Can suffer from "dead neurons" (x always < 0 => gradient = 0).
- Preferred default for hidden layers in deep networks.

### Sigmoid
```
f(x) = 1 / (1 + exp(-x))
Range: (0, 1)
```
- Output is always positive, which can bias later layers.
- **Saturates** for large |x|: gradient approaches 0 — this slows learning (vanishing gradients).
- Still used in binary classification output layers.

### Tanh (Hyperbolic Tangent)
```
f(x) = (exp(x) - exp(-x)) / (exp(x) + exp(-x))
Range: (-1, 1)
```
- Zero-centered (unlike Sigmoid), which helps gradients flow.
- Still **saturates** for large |x|.
- Stronger gradient than Sigmoid near 0.

---
## 2. Imports and Configuration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# --- Shared hyperparameters (identical for all experiments) ---
HIDDEN_1   = 128
HIDDEN_2   = 64
EPOCHS     = 10    # Increase to 20 for better differentiation
BATCH_SIZE = 64
VAL_SPLIT  = 0.1
# --------------------------------------------------------------

print(f"TensorFlow : {tf.__version__}")
print(f"Epochs     : {EPOCHS}  |  Batch size: {BATCH_SIZE}")

---
## 3. Plot Activation Function Curves

In [ ]:
x = np.linspace(-5, 5, 400)

def relu(x):    return np.maximum(0, x)
def sigmoid(x): return 1 / (1 + np.exp(-x))
def tanh(x):    return np.tanh(x)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fns   = [relu, sigmoid, tanh]
names = ["ReLU", "Sigmoid", "Tanh"]
colors = ["steelblue", "tomato", "seagreen"]

for ax, fn, name, color in zip(axes, fns, names, colors):
    ax.plot(x, fn(x), color=color, linewidth=2.5)
    ax.axhline(0, color="gray", linewidth=0.8, linestyle="--")
    ax.axvline(0, color="gray", linewidth=0.8, linestyle="--")
    ax.set_title(name, fontsize=14)
    ax.set_xlabel("Input (x)")
    ax.set_ylabel("Output f(x)")
    ax.grid(True, linestyle="--", alpha=0.4)

plt.suptitle("Activation Function Curves", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

---
## 4. Load and Prepare MNIST (shared across all experiments)

In [ ]:
(X_train_raw, y_train), (X_test_raw, y_test) = keras.datasets.mnist.load_data()

# Normalize to [0, 1]
X_train = X_train_raw.astype("float32") / 255.0
X_test  = X_test_raw.astype("float32")  / 255.0

print(f"Training: {X_train.shape}  Test: {X_test.shape}")

---
## 5. Model Builder

In [ ]:
def build_model(activation):
    """Build a 3-Dense-layer model with the given hidden activation."""
    tf.random.set_seed(SEED)
    model = keras.Sequential([
        keras.layers.Flatten(input_shape=(28, 28)),
        keras.layers.Dense(HIDDEN_1, activation=activation),
        keras.layers.Dense(HIDDEN_2, activation=activation),
        keras.layers.Dense(10, activation="softmax"),   # Output always softmax
    ])
    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

---
## 6. Run Experiments

Each activation function trains an identical architecture. Only the hidden-layer activation changes.

In [ ]:
activations = ["relu", "sigmoid", "tanh"]
histories   = {}
test_results = {}

for act in activations:
    print(f"\n{'='*50}")
    print(f"Training with activation: {act.upper()}")
    print(f"{'='*50}")
    model = build_model(act)
    hist = model.fit(
        X_train, y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_split=VAL_SPLIT,
        verbose=1
    )
    loss, acc = model.evaluate(X_test, y_test, verbose=0)
    histories[act]    = hist.history
    test_results[act] = {"loss": loss, "accuracy": acc}
    print(f"  Test accuracy ({act}): {acc * 100:.2f}%  |  Test loss: {loss:.4f}")

---
## 7. Compare Training and Validation Curves

In [ ]:
act_colors = {"relu": "steelblue", "sigmoid": "tomato", "tanh": "seagreen"}
epoch_range = range(1, EPOCHS + 1)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

metrics = [
    ("accuracy",     "val_accuracy",  "Training Accuracy",     "Validation Accuracy"),
    ("loss",         "val_loss",      "Training Loss",         "Validation Loss"),
]

for row, (train_key, val_key, train_label, val_label) in enumerate(metrics):
    ax_train = axes[row, 0]
    ax_val   = axes[row, 1]
    for act in activations:
        h = histories[act]
        ax_train.plot(epoch_range, h[train_key],
                      label=act.upper(), color=act_colors[act], linewidth=2)
        ax_val.plot(epoch_range, h[val_key],
                    label=act.upper(), color=act_colors[act], linewidth=2, linestyle="--")
    ax_train.set_title(train_label, fontsize=12)
    ax_train.set_xlabel("Epoch"); ax_train.set_ylabel(train_label.split()[-1])
    ax_train.legend(); ax_train.grid(True, linestyle="--", alpha=0.4)
    ax_val.set_title(val_label, fontsize=12)
    ax_val.set_xlabel("Epoch"); ax_val.set_ylabel(val_label.split()[-1])
    ax_val.legend(); ax_val.grid(True, linestyle="--", alpha=0.4)

plt.suptitle("Activation Function Comparison on MNIST", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
## 8. Summary Table

In [ ]:
print(f"\n{'Activation':<12} {'Final Train Acc':<18} {'Final Val Acc':<16} {'Test Acc':<12} {'Test Loss'}")
print("-" * 75)
for act in activations:
    h = histories[act]
    train_acc = h["accuracy"][-1]     * 100
    val_acc   = h["val_accuracy"][-1] * 100
    test_acc  = test_results[act]["accuracy"] * 100
    test_loss = test_results[act]["loss"]
    print(f"{act.upper():<12} {train_acc:<18.2f} {val_acc:<16.2f} {test_acc:<12.2f} {test_loss:.4f}")

---
## 9. Observations and Conclusion

### Expected Patterns (based on actual results above)

- **ReLU** typically converges fastest. It does not saturate for positive inputs, so gradients flow freely. For MNIST with a shallow network, all activations perform well, but ReLU usually leads from epoch 1.
- **Sigmoid** often starts slower (especially in early epochs) because its gradient is small everywhere except near 0. In deeper networks this worsens into the vanishing gradient problem.
- **Tanh** is zero-centered (unlike Sigmoid), so gradients flow a little better and it often performs comparably to ReLU on shallow problems.

### Key Limitations

| Activation | Advantage | Limitation |
|------------|-----------|------------|
| ReLU       | Fast convergence, no saturation for x>0 | Dead neurons (x always negative) |
| Sigmoid    | Probabilistic output in (0,1) | Saturates; vanishing gradients |
| Tanh       | Zero-centered, stronger gradient than Sigmoid | Still saturates; slightly slower than ReLU |

**Note:** On a simple 2-layer network applied to MNIST, differences are modest. The gap between activations becomes far more pronounced in deep networks with many layers.